# U7 Test-time steering 大挑戰

你有：一個訓練好的 1D flow matching 模型（參數不動）和一個公開的 reward $R(x)$。
目標分布：$p_R(x) \propto p_{\text{model}}(x)\,e^{\lambda R(x)}$，$\lambda = 3$。

請繳交：**2,000 個一維的點**，存成 `submission.csv`（兩欄 `ID,x`）。
Leaderboard：你的 2,000 個點的經驗分布到 $p_R$ 的 Wasserstein-1 distance，越小越好，理想上最小是0（用抽樣的狀況去算 $p_R$ 自己對自己可以到 0.006）。
預算：每次提交最多 **200,000 次網路呼叫**（`NFE` 會自動累計；原樣取樣 2,000 點 × 100 步 = 200,000）。

規則：不能動模型的參數；提交的點必須來自模型的取樣後的結果；方法用的 NFE 不要超過限制。

## 1. 下載／載入 checkpoint
這一格會依序找：Kaggle 競賽的 Data、同目錄的 `fm_model.pt`、最後從 `CKPT_URL`（GitHub）下載。
在 Colab 或自己的電腦上直接跑就好，不用改任何東西。

In [ ]:
import math, glob, os, urllib.request
import numpy as np, torch, torch.nn as nn

CKPT_URL = "https://raw.githubusercontent.com/shihhsinwang0214/test-time-steering-lab/main/fm_model.pt"   # 老師把 GITHUB_USER 換掉
path = (glob.glob("/kaggle/input/**/fm_model.pt", recursive=True) or ["fm_model.pt"])[0]
if not os.path.exists(path):
    print("downloading checkpoint from", CKPT_URL)
    urllib.request.urlretrieve(CKPT_URL, "fm_model.pt"); path = "fm_model.pt"

class TimeEmb(nn.Module):
    def __init__(self, dim=64):
        super().__init__(); self.dim = dim
    def forward(self, t):
        half = self.dim // 2
        freqs = torch.exp(-math.log(1000.0) * torch.arange(half, dtype=torch.float32) / half)
        ang = t[:, None] * 1000.0 * freqs[None, :]
        return torch.cat([ang.sin(), ang.cos()], 1)

class FlowNet(nn.Module):
    def __init__(self, hidden=256, temb=64):
        super().__init__()
        self.temb = TimeEmb(temb)
        self.net = nn.Sequential(nn.Linear(1 + temb, hidden), nn.SiLU(),
                                 nn.Linear(hidden, hidden), nn.SiLU(),
                                 nn.Linear(hidden, hidden), nn.SiLU(),
                                 nn.Linear(hidden, 1))
    def forward(self, x, t):
        return self.net(torch.cat([x, self.temb(t)], 1))

model = FlowNet().eval()
model.load_state_dict(torch.load(path, map_location="cpu"))
for p in model.parameters():
    p.requires_grad_(False)
print("loaded", path)

## 2. 模型提供的東西

慣例：$t$ 從 0（純噪聲 $z\sim\mathcal N(0,1)$）走到 1（資料）；$x_t=(1-t)z+t\,x$。

| 名字 | 意思 | 花 NFE？ |
|---|---|---|
| `v_theta(x_t, t)` | velocity field $v_\theta(x_t,t)$，形狀 `(n,1)` 進 `(n,1)` 出 | 是（+n） |
| `x_hat(x_t, t, v)` | 去噪估計 $\hat x = x_t + (1-t)\,v$：模型對終點的最佳猜測（U7 的 $\hat x_0$） | 否（用算好的 v） |
| `sample(n, steps)` | 最普通的 Euler 取樣；它的迴圈就在下面，複製去改 | 是（n×steps） |
| `NFE` | 累計的網路呼叫次數 | — |

In [ ]:
NFE = 0

def v_theta(x_t, t):
    """x_t: (n,1) tensor；t: float 或 (n,) tensor。要對 x_t 求 gradient 就傳 requires_grad 的 x_t。"""
    global NFE
    x_t = torch.as_tensor(x_t, dtype=torch.float32).reshape(-1, 1)
    tt = torch.as_tensor(t, dtype=torch.float32).reshape(-1).expand(x_t.shape[0]) if np.ndim(t) == 0 else torch.as_tensor(t, dtype=torch.float32)
    NFE += x_t.shape[0]
    return model(x_t, tt)

def x_hat(x_t, t, v):
    return x_t + (1 - t) * v

@torch.no_grad()
def sample(n=2000, steps=100, z=None):
    x = torch.randn(n, 1) if z is None else torch.as_tensor(z, dtype=torch.float32).reshape(-1, 1).clone()
    ts = torch.linspace(0, 1, steps + 1)
    for i in range(steps):
        t, dt = float(ts[i]), float(ts[i + 1] - ts[i])
        v = v_theta(x, t)            # 算 NFE
        xh = x_hat(x, t, v)          # 途中就能看的去噪估計，不另外花 NFE
        x = x + dt * v               # Euler 一步
    return x.numpy()

## 3. Reward（公開、閉式，可以反傳）

$$R(x)=2e^{-\frac{(x+7.2)^2}{2(0.3)^2}}+4e^{-\frac{(x+4.2)^2}{2(0.3)^2}}+3e^{-\frac{(x+1.3)^2}{2(0.3)^2}}+5e^{-\frac{(x-4.3)^2}{2(0.3)^2}}+e^{-\frac{(x-7.2)^2}{2(0.3)^2}}$$

In [ ]:
LAM = 3.0
PEAKS = torch.tensor([[-7.2, 2.0], [-4.2, 4.0], [-1.3, 3.0], [4.3, 5.0], [7.2, 1.0]])   # (中心, 高度)，sd 都是 0.3

def reward(x):
    """接受 numpy 或 tensor（(n,1) 或 (n,)），回 (n,) 同型別。tensor 進去可以對它求 gradient。"""
    is_np = not isinstance(x, torch.Tensor)
    xt = torch.as_tensor(np.asarray(x) if is_np else x, dtype=torch.float32).reshape(-1, 1)
    r = (PEAKS[:, 1] * torch.exp(-(xt - PEAKS[:, 0]) ** 2 / (2 * 0.3 ** 2))).sum(1)
    return r.numpy().astype(float) if is_np else r

## 4. 先看一眼：什麼都不做的樣子
直方圖是模型的樣本，橘線是 $R(x)$。問自己：reward 最高的地方，模型有多少質量？

In [ ]:
import matplotlib.pyplot as plt

def show(x, title=""):
    x = np.asarray(x).reshape(-1)
    r = reward(x)
    print(f"n={len(x)}  mean reward={r.mean():.3f}  NFE so far={NFE:,}")
    fig, ax = plt.subplots(figsize=(8, 3)); ax2 = ax.twinx()
    g = np.linspace(-9, 9, 1801)
    ax2.plot(g, reward(g), color="#c2410c", lw=1.3); ax2.set_ylabel("R(x)", color="#c2410c"); ax2.set_ylim(0, 5.6)
    ax.hist(x, bins=np.linspace(-9, 9, 361), density=True, color="#4cc9f0", alpha=.8)
    ax.set_xlim(-9, 9); ax.set_ylabel("density"); ax.set_title(title); plt.show()

NFE = 0
xs = sample(2000, steps=100)
show(xs, "no steering")

## 5. 你的方法（填這一格）

目標是讓交出去的 2,000 個點像是從 $p_R \propto p_{\text{model}}\,e^{\lambda R}$ 抽出來的。
怎麼做沒有規定：可以在途中改 `v`、途中看 `x_hat` 決定留誰、跑完整段再挑或加權、
挑完放回噪聲（`(1-t)*torch.randn_like(x) + t*x`）再走幾輪、固定起點 `z` 去搜尋……

你手上有：`v_theta`、`x_hat`、`sample`、`reward`、`LAM`、`NFE`，還有上面那個可以複製去改的迴圈。
寫之前先回答三個問題：這一步對一個樣本做了什麼？你用到的量現在真的能算，還是你在估它？NFE 花在哪？

In [ ]:
def my_method(n=2000):
    # TODO：回傳 (n, 1) 或 (n,) 的 numpy array
    raise NotImplementedError

# NFE = 0
# xs = my_method()
# show(xs, "my method")

## 6. 輸出格式
`submission.csv`，兩欄：`ID`（0..1999）、`x`（浮點數）。恰好 2,000 列，不能有 NaN。
在 Kaggle 上從這個 notebook 的輸出提交（Save Version → Output → Submit）。
方法卡上填：`show()` 最後印出的 **mean reward** 與 **NFE**。

In [ ]:
import pandas as pd

def make_submission(x, path="submission.csv"):
    x = np.asarray(x, dtype=float).reshape(-1)
    assert x.shape == (2000,), f"需要恰好 2000 個點，拿到 {x.shape}"
    assert np.isfinite(x).all(), "有 NaN 或 inf"
    pd.DataFrame({"ID": np.arange(2000), "x": x}).to_csv(path, index=False)
    print("saved", path)

make_submission(xs)